#### TRANSFORM JOIN BETWEEN `GIZMO.SILVER.CUSTOMERS` AND `GIZMO.SILVER.ADDRESSES`

#### WRITE TRANSFORMED DATA TO GOLD SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: GOLD
3. TABLE NAME: CUSTOMERS_ORDERS

In [0]:
%python
cust_addr_df = spark.sql('''SELECT
cust.customer_id,
cust.first_name,
cust.last_name,
cust.date_of_birth,
cust.email,
cust.telephone,
cust.member_since,
addr.shipping_address_line_1 AS shipping_address_line_1,
addr.billing_address_line_1 AS billing_address_line_1,
addr.billing_city AS billing_city,
addr.billing_state AS billing_state,
addr.shipping_postcode AS shipping_postcode,
addr.shipping_state AS shipping_address,
addr.billing_postcode AS billing_postcode
FROM
GIZMO.SILVER.CUSTOMERS CUST INNER JOIN GIZMO.SILVER.ADDRESSES ADDR
ON cust.customer_id = addr.customer_id''')
print(f'Rows Effected: {cust_addr_df.count()}')

In [0]:
CREATE OR REPLACE TABLE GIZMO.GOLD.CUSTOMERS_ORDERS
AS
SELECT
cust.customer_id,
cust.first_name,
cust.last_name,
cust.date_of_birth,
cust.email,
cust.telephone,
cust.member_since,
addr.shipping_address_line_1 AS shipping_address_line_1,
addr.billing_address_line_1 AS billing_address_line_1,
addr.billing_city AS billing_city,
addr.billing_state AS billing_state,
addr.shipping_postcode AS shipping_postcode,
addr.shipping_state AS shipping_address,
addr.billing_postcode AS billing_postcode
FROM
GIZMO.SILVER.CUSTOMERS CUST INNER JOIN GIZMO.SILVER.ADDRESSES ADDR
ON cust.customer_id = addr.customer_id;

In [0]:
%python
cust_addr_count_df = spark.sql('''SELECT * FROM GIZMO.GOLD.CUSTOMERS_ORDERS''');
print(f'Row Count: {cust_addr_count_df.count()}')

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config

In [0]:
%python
try:
    verify_pipeline_counts(cust_addr_df, cust_addr_count_df, "SQL-07.TransformCustomersAddresses")
except AssertionError as e:
    raise

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

load_start_time = datetime.now()
pipeline_name = 'SQL-07.TransformCustomersAddresses'
status = "SUCCESS"
message = "Loaded Orders data into bronze view"
record_count = 0

try:
    count_df = spark.sql('''SELECT
                                cust.customer_id,
                                cust.first_name,
                                cust.last_name,
                                cust.date_of_birth,
                                cust.email,
                                cust.telephone,
                                cust.member_since,
                                addr.shipping_address_line_1 AS shipping_address_line_1,
                                addr.billing_address_line_1 AS billing_address_line_1,
                                addr.billing_city AS billing_city,
                                addr.billing_state AS billing_state,
                                addr.shipping_postcode AS shipping_postcode,
                                addr.shipping_state AS shipping_address,
                                addr.billing_postcode AS billing_postcode
                                FROM
                                GIZMO.SILVER.CUSTOMERS CUST INNER JOIN GIZMO.SILVER.ADDRESSES ADDR
                                ON cust.customer_id = addr.customer_id''') 
    record_count = count_df.count()
except Exception as e:
    status = "FAILED"
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1

finally:
    load_end_time = datetime.now()
    current_date = load_end_time.date()
    try:
        max_run_df = spark.table("GIZMO.AUDIT.AUDIT_LOGS") \
            .filter(
                (F.col("event_time") == F.lit(current_date)) & 
                (F.col("pipeline_name") == F.lit(pipeline_name))
            ) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception:
        next_run_int = 1
    run_id_str = f"{next_run_int:02d}"
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table="GIZMO.SILVER.CUSTOMERS, GIZMO.SILVER.ADDRESSES",
        target_table="GIZMO.GOLD.CUSTOMERS_ORDERS",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])
    log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
    log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.AUDIT.AUDIT_LOGS")
    print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")
    if status == "FAILED":
        raise RuntimeError(message)
# Undefined name `refunds_df` is not present in this code, so no action needed.

#### QUERY AND VALIDATE THE RECORDS
1.  `GIZMO.GOLD.filter_state_fn` filter the records based on state.
2.  `GIZMO.GOLD.mask_billing_address_line_1_fn`,`GIZMO.GOLD.email_fn` mask the column billing_address_line_1 & email
3.   FILTER THE ROWS BASED ON `GIZMO.GOLD.mask_billing_address_line_1_fn`,`GIZMO.GOLD.email_fn`

In [0]:
SELECT * FROM GIZMO.GOLD.CUSTOMERS_ORDERS ORDER BY 1;

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS ADDRESSES LOADED INTO GIZMO.GOLD.CUSTOMERS_ADDRESS")